# S1.9-G3 - Gold Layer

## Objectif

L'objectif de cette étape est de construire la couche Gold du Data Lake AgriLens.

La Gold Layer représente la version finale, validée et prête à être utilisée par les futurs pipelines Machine Learning, Deep Learning et MLOps.

Elle regroupe dans un unique dataset les informations issues de l'inventaire(inventoryV3), du split officiel(splitmanifest) et du Class Mapping.

Cette couche constitue la source officielle de vérité du projet.

In [2]:
import pandas as pd

# Chargement des artefacts validés

Nous chargeons les trois artefacts officiels validés durant la phase S1 :

- inventory_v3.csv
- split_manifest_v2.csv
- class_mapping.csv

Ces fichiers serviront à construire le Gold Dataset.

In [3]:
inventory = pd.read_csv(
    "../../01_data/metadata/inventory_v3.csv"
)

split = pd.read_csv(
    "../../01_data/metadata/split_manifest_v2.csv"
)

mapping = pd.read_csv(
    "../../01_data/metadata/class_mapping.csv"
)

In [6]:
print("inventory:", inventory.shape)
print("split:",split.shape)
print("mapping:",mapping.shape)

inventory: (26397, 20)
split: (26388, 2)
mapping: (17, 2)


# Intégration du split officiel

L'objectif de cette étape est de conserver uniquement les images présentes dans le split officiel validé.

Le fichier split_manifest_v2.csv constitue désormais la référence après correction des fuites MD5 détectées lors du contrôle post-split.

Cette fusion permet également d'associer à chaque image son ensemble d'appartenance :

- train
- validation
- test

In [8]:
gold= inventory.merge(
    split,
    on="filepath",
    how="inner",
    suffixes=("", "_final")
)
gold.shape

(26388, 21)

# Intégration du Class Mapping

L'objectif de cette étape est d'associer à chaque image son identifiant numérique officiel.

Le fichier class_mapping.csv constitue la référence officielle reliant chaque nom de classe à un identifiant unique.

Après cette fusion, chaque image possédera :

- son label textuel
- son identifiant numérique

Le Gold Dataset sera ainsi directement exploitable par les futurs modèles de Machine Learning et de Deep Learning.

In [10]:
gold=gold.merge(
    mapping,
    left_on="classe",
    right_on="class_name",
    how="left"
)
gold.shape
    

(26388, 23)

# Vérification du Class Mapping

L'objectif de cette étape est de vérifier que chaque image possède un identifiant de classe valide.

Une absence de correspondance entre les classes du dataset et le Class Mapping entraînerait la présence de valeurs manquantes.

La vérification permet de confirmer que toutes les classes ont correctement été associées à leur identifiant numérique.

In [11]:
gold[["classe", "class_id"]].head()

,classe,class_id
0,black_pod_rot,14
1,black_pod_rot,14
2,black_pod_rot,14
3,black_pod_rot,14
4,black_pod_rot,14


In [12]:
gold["class_id"].isna().sum()

np.int64(0)

# Export du Gold Dataset

L'objectif de cette étape est de sauvegarder la version finale du dataset dans la couche Gold du Data Lake.

Cette version regroupe :

- les métadonnées de l'inventaire ;
- le split officiel validé ;
- le Class Mapping officiel.

Le fichier produit constituera la source de vérité utilisée lors des futures phases Machine Learning, Deep Learning et MLOps.

In [14]:
gold.to_csv("../../01_data/lake/gold/gold_dataset.parquet",
            index=False
           )
print("ok")

ok


In [16]:
df=pd.read_csv("../../01_data/lake/gold/gold_dataset.parquet")
df.shape

(26388, 23)

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26388 entries, 0 to 26387
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   source           26388 non-null  object 
 1   culture          26388 non-null  object 
 2   split            26388 non-null  object 
 3   classe           26388 non-null  object 
 4   filename         26388 non-null  object 
 5   filepath         26388 non-null  object 
 6   extension        26388 non-null  object 
 7   format           26388 non-null  object 
 8   width            26388 non-null  int64  
 9   height           26388 non-null  int64  
 10  mode             26388 non-null  object 
 11  channels         26388 non-null  int64  
 12  problem          26388 non-null  object 
 13  ratio            26388 non-null  float64
 14  brightness_mean  26388 non-null  float64
 15  exposure_status  26388 non-null  object 
 16  md5_hash         26388 non-null  object 
 17  duplicate_ex

In [18]:
df.head()

,source,culture,split,classe,filename,filepath,extension,format,width,height,...,ratio,brightness_mean,exposure_status,md5_hash,duplicate_exact,leakage_risk,phash,split_final,class_id,class_name
0,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_1.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,...,1.0,102.669785,normale,3cf3901bd20e3910ce2bd2a34285e151,False,False,da95a424acb2f2d3,train,14,black_pod_rot
1,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_10.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,...,1.0,121.778247,normale,2473be012438c8db2a0c1f454c226928,False,False,eef242f7a310b18a,test,14,black_pod_rot
2,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_100.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,...,1.0,120.446264,normale,af88d5ff63102933d009e06602aa9aa8,False,False,d1f58ff12f01ba04,train,14,black_pod_rot
3,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_101.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,...,1.0,139.776007,normale,d33754ff3dc4621047583561bc442850,False,False,a116ee225d6a496f,train,14,black_pod_rot
4,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_102.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,...,1.0,125.152757,normale,d7b2a39d2eac0a869f5510581943bed2,False,False,eeec88a22f1db215,train,14,black_pod_rot
